# 15. Ereignisse und Zufall: Abläufe simulieren und Wahrscheinlichkeiten schätzen

Manche Aufgaben beschreiben einen **Ablauf in der Zeit**: Kunden kommen an und werden bedient, Aufträge treffen in einer Werkstatt ein, Kristalle wachsen. Gefragt ist dann zum Beispiel,
wie lange jemand warten muss. Das kann man oft nicht ausrechnen, aber man kann den Ablauf **nachspielen** (simulieren). Andere Aufgaben fragen nach einer **Wahrscheinlichkeit**
(„Wie oft gewinnt der Beste?“). Auch die kann man durch **Nachspielen mit Zufallszahlen** schätzen.

Du kennst die Bausteine schon: die Prioritätswarteschlange `heapq` aus [Notebook 7](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/07_dijkstra.ipynb) und das Zählen aller Fälle aus
[Notebook 1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/01_durchprobieren.ipynb). Hier setzt du sie für Simulationen ein.

In diesem Notebook lernst du:

- warum man nicht **jede Minute** simuliert, sondern von **Ereignis zu Ereignis springt** (mit einem **Zähler** der Schritte als Beweis),
- eine **Ereignisliste** mit `heapq` und Tupeln `(zeit, art, nummer)` aufzubauen und zu verstehen, was bei **gleicher Zeit** passiert,
- **Kennzahlen** wie mittlere und größte Wartezeit auszurechnen und Varianten (mehr Schalter, andere Reihenfolge) zu vergleichen,
- mit `random.seed` **nachvollziehbare Zufallszahlen** zu erzeugen und das **Monte-Carlo-Verfahren** anzuwenden: Zufallsexperiment oft wiederholen und die Häufigkeit auswerten,
- zu sehen, wie die **Schwankung** der Schätzung mit der **Zahl der Wiederholungen** abnimmt, und eine Schätzung mit der **genauen Wahrscheinlichkeit** zu vergleichen.

In den Wettbewerbs-Ecken simulierst du eine Fahrradwerkstatt und ein Turnier (beides vereinfacht), und zum Schluss lässt du Kristalle wachsen.

## 1. Warum nicht jede Minute simulieren?

Am **Schalter** eines Amtes kommen Kunden an. Jeder Kunde hat einen **Ankunftszeitpunkt** und eine **Bedienzeit** (beides in Minuten ab Öffnung). Es gibt einen Schalter. Wer ankommt, wenn der
Schalter besetzt ist, **wartet**. Bedient wird in der Reihenfolge der Ankunft. Die **Wartezeit** eines Kunden ist die Zeit von seiner Ankunft bis zum Beginn seiner Bedienung.

Die Kunden stehen als Liste von Tupeln `(ankunft, dauer)`, sortiert nach Ankunft:

```python
kunden = [(10, 12), (25, 8), (30, 15), (95, 10), (100, 6), (240, 9), (250, 5)]
```

Der Kunde `(30, 15)` kommt in Minute 30 an. Der Schalter ist von Kunde `(25, 8)` besetzt, der in Minute 25 begann und in Minute 33 fertig ist. Er wartet also 3 Minuten.

Die naheliegende Simulation lässt die **Uhr Minute für Minute laufen** und fragt in jeder Minute: Ist der nächste Kunde da, und ist der Schalter frei? Das Beispiel unten tut das
und zählt in `schritte`, wie viele Minuten es dafür durchläuft.

In [ ]:
kunden = [(10, 12), (25, 8), (30, 15), (95, 10), (100, 6), (240, 9), (250, 5)]

def minutenweise(kunden):
    wartezeiten = []
    frei_ab = 0                  # ab dieser Minute ist der Schalter frei
    naechster = 0                # Nummer des nächsten Kunden, der bedient wird
    t = 0                        # die Uhr
    schritte = 0
    while naechster < len(kunden):
        schritte = schritte + 1
        ankunft, dauer = kunden[naechster]
        if ankunft <= t and frei_ab <= t:          # Kunde ist da und der Schalter ist frei
            wartezeiten.append(t - ankunft)
            frei_ab = t + dauer
            naechster = naechster + 1
        t = t + 1
    return wartezeiten, schritte

wartezeiten, schritte = minutenweise(kunden)
print(wartezeiten)
print(schritte, 'Minuten simuliert')

Die Wartezeiten sind `[0, 0, 3, 0, 5, 0, 0]`. Dafür wurden **251 Minuten** durchlaufen, obwohl nur 7 Kunden da sind und in den allermeisten Minuten **nichts passiert**
(zwischen Minute 100 und 240 zum Beispiel gar nichts). Es passiert nur etwas, wenn ein Kunde **ankommt** oder **fertig wird**. Diese Zeitpunkte heißen **Ereignisse**.
Man kann die Uhr also direkt von Ereignis zu Ereignis **springen** lassen.

#### Aufgabe 1.1

Schreibe `springen(kunden)`. Sie gibt wie `minutenweise` ein Paar `(wartezeiten, schritte)` zurück, rechnet aber **ohne Uhr**: Für jeden Kunden (ein Schritt) ist der **Beginn seiner
Bedienung** der spätere der beiden Zeitpunkte „er kommt an“ und „der Schalter wird frei“ (`max`). Seine Wartezeit ist Beginn minus Ankunft. Danach ist der Schalter ab `beginn + dauer` frei.

**Erwartete Ausgabe:**
```
[0, 0, 3, 0, 5, 0, 0]
7 Schritte
```

In [ ]:
def springen(kunden):
    wartezeiten = []
    frei_ab = 0
    schritte = 0
    # dein Code
    return wartezeiten, schritte

kunden = [(10, 12), (25, 8), (30, 15), (95, 10), (100, 6), (240, 9), (250, 5)]
wartezeiten, schritte = springen(kunden)
print(wartezeiten)
print(schritte, 'Schritte')

<details>
<summary>Tipp anzeigen</summary>

Schleife `for ankunft, dauer in kunden:`. Darin `beginn = max(ankunft, frei_ab)`, dann die Wartezeit eintragen und `frei_ab` neu setzen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def springen(kunden):
    wartezeiten = []
    frei_ab = 0
    schritte = 0
    for ankunft, dauer in kunden:
        schritte = schritte + 1
        beginn = max(ankunft, frei_ab)            # später von beiden: Kunde da, Schalter frei
        wartezeiten.append(beginn - ankunft)
        frei_ab = beginn + dauer
    return wartezeiten, schritte

kunden = [(10, 12), (25, 8), (30, 15), (95, 10), (100, 6), (240, 9), (250, 5)]
wartezeiten, schritte = springen(kunden)
print(wartezeiten)
print(schritte, 'Schritte')
```

Dieselben Wartezeiten wie bei der Minutensimulation, aber mit **7 Schritten statt 251**. Die Zahl der Schritte hängt nur von der **Zahl der Kunden** ab, nicht davon, wie lange der Tag dauert.

</details>

#### Frage 1.2

1. Statt in Minuten zählt jemand die Zeit in **Sekunden** (alle Zahlen mal 60). Wie viele Schritte braucht `minutenweise` jetzt ungefähr, wie viele `springen`?
2. In der Fahrradwerkstatt (Ecke 1) liegen die Eingangszeiten der Aufträge bei mehreren Hunderttausend Minuten, es gibt aber nur ein paar Hundert Aufträge. Welche Art Simulation lohnt sich dort?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `minutenweise` braucht jetzt rund 251 · 60 = **15 060** Schritte (jede Sekunde ein Schritt), `springen` weiterhin **7**.
2. Die mit **Sprüngen von Ereignis zu Ereignis**: Ein paar Hundert Schritte statt mehrerer Hunderttausend. Die Laufzeit hängt dann von der **Zahl der Ereignisse** ab (nicht von der Länge der Zeit).

</details>

## 2. Die Ereignisliste mit `heapq`

Mit **einem** Schalter ging es noch ohne Liste. Bei **mehreren Schaltern** (oder mehreren Mechanikern) wird es schwerer: Wer wird als Nächster fertig? Man braucht eine **Ereignisliste**:
In ihr stehen alle Ereignisse, die noch kommen, und man nimmt immer das mit der **kleinsten Zeit** als Nächstes. Das ist die **Prioritätswarteschlange** aus [Notebook 7](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/07_dijkstra.ipynb).
Kurz zur Erinnerung (`from heapq import heappush, heappop`):

- `heappush(liste, eintrag)` fügt einen Eintrag ein,
- `heappop(liste)` entfernt den **kleinsten** Eintrag und gibt ihn zurück; `len(liste)` zeigt, wie viele noch drin sind.

Ein Ereignis ist ein Tupel `(zeit, art, nummer)`: `zeit` ist der Zeitpunkt, `art` sagt, was passiert (**0 = Kunde wird fertig**, **1 = Kunde kommt an**), `nummer` ist die Nummer des Kunden.
Tupel werden **der Reihe nach** verglichen: erst die Zeit, bei gleicher Zeit die Art, bei gleicher Art die Nummer. So ist jede Reihenfolge **eindeutig festgelegt**, auch bei gleichzeitigen Ereignissen.
Hier ist `0` vor `1` gewählt: Wird ein Kunde fertig in derselben Minute, in der ein neuer ankommt, kommt zuerst das Fertig.

#### Aufgabe 2.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
from heapq import heappush, heappop

ereignisse = []
heappush(ereignisse, (30, 1, 4))      # Kunde 4 kommt in Minute 30 an
heappush(ereignisse, (12, 0, 1))      # Kunde 1 wird in Minute 12 fertig
heappush(ereignisse, (30, 0, 2))      # Kunde 2 wird in Minute 30 fertig
heappush(ereignisse, (30, 1, 3))      # Kunde 3 kommt in Minute 30 an
print(heappop(ereignisse))
print(heappop(ereignisse))
print(heappop(ereignisse))
print(len(ereignisse))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
(12, 0, 1)
(30, 0, 2)
(30, 1, 3)
1
```

Zuerst kommt Minute 12. Bei Minute 30 gibt es drei Ereignisse: `(30, 0, 2)` hat die kleinste Art (0 = fertig) und kommt zuerst, danach `(30, 1, 3)` vor `(30, 1, 4)`, weil die Nummer 3 kleiner
als 4 ist. Nach drei Entnahmen bleibt ein Ereignis in der Liste.

</details>

#### Aufgabe 2.2

Schreibe `simuliere_schalter(kunden, anzahl)` für `anzahl` Schalter. Sie gibt `(wartezeiten, ereignisse)` zurück: die Wartezeit jedes Kunden (in der Reihenfolge der Kundenliste) und die Zahl der
bearbeiteten Ereignisse. Das Gerüst legt die Ankünfte schon in die Ereignisliste. Du schreibst die Schleife: Hole das nächste Ereignis. Bei einer **Ankunft** stellt sich der Kunde in die
Warteschlange `wartend`, bei **Fertig** wird ein Schalter frei. Danach gilt: **Solange ein Schalter frei ist und jemand wartet**, nimmt der erste Wartende den Schalter. Sein **Beginn** ist die
aktuelle Zeit, die Wartezeit Beginn minus Ankunft, und sein **Fertig-Ereignis** `(beginn + dauer, 0, nummer)` kommt in die Ereignisliste.

**Erwartete Ausgabe:**
```
[0, 0, 3, 3, 0]
10 Ereignisse
```

In [ ]:
from heapq import heappush, heappop

FERTIG = 0
ANKUNFT = 1

def simuliere_schalter(kunden, anzahl):
    ereignisliste = []
    for nummer in range(len(kunden)):
        heappush(ereignisliste, (kunden[nummer][0], ANKUNFT, nummer))
    frei = anzahl                       # Zahl der freien Schalter
    wartend = []                        # Nummern der wartenden Kunden, der älteste vorne
    wartezeiten = [0] * len(kunden)
    ereignisse = 0
    while len(ereignisliste) > 0:
        zeit, art, nummer = heappop(ereignisliste)
        ereignisse = ereignisse + 1
        # dein Code
    return wartezeiten, ereignisse

kunden = [(0, 5), (1, 5), (2, 5), (3, 2), (10, 4)]
wartezeiten, ereignisse = simuliere_schalter(kunden, 2)
print(wartezeiten)
print(ereignisse, 'Ereignisse')

<details>
<summary>Tipp anzeigen</summary>

Zwei Fälle mit `if art == ANKUNFT:` / `else:`. Danach (nicht eingerückt unter dem `else`) eine `while`-Schleife mit der Bedingung `frei > 0 and len(wartend) > 0`. Darin `wartend.pop(0)` (entfernt den ersten
Eintrag der Liste und gibt ihn zurück), `frei` um 1 verkleinern, `wartezeiten[...]` setzen und mit `heappush` das Fertig-Ereignis einfügen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from heapq import heappush, heappop

FERTIG = 0
ANKUNFT = 1

def simuliere_schalter(kunden, anzahl):
    ereignisliste = []
    for nummer in range(len(kunden)):
        heappush(ereignisliste, (kunden[nummer][0], ANKUNFT, nummer))
    frei = anzahl                       # Zahl der freien Schalter
    wartend = []                        # Nummern der wartenden Kunden, der älteste vorne
    wartezeiten = [0] * len(kunden)
    ereignisse = 0
    while len(ereignisliste) > 0:
        zeit, art, nummer = heappop(ereignisliste)
        ereignisse = ereignisse + 1
        if art == ANKUNFT:
            wartend.append(nummer)
        else:
            frei = frei + 1
        while frei > 0 and len(wartend) > 0:         # Schalter frei und jemand wartet
            naechster = wartend.pop(0)               # der Älteste kommt dran
            frei = frei - 1
            wartezeiten[naechster] = zeit - kunden[naechster][0]
            heappush(ereignisliste, (zeit + kunden[naechster][1], FERTIG, naechster))
    return wartezeiten, ereignisse

kunden = [(0, 5), (1, 5), (2, 5), (3, 2), (10, 4)]
wartezeiten, ereignisse = simuliere_schalter(kunden, 2)
print(wartezeiten)
print(ereignisse, 'Ereignisse')
```

Zwei Schalter, fünf Kunden: Kunde 0 und 1 beginnen sofort (Minute 0 und 1). Kunde 2 kommt in Minute 2 und muss bis Minute 5 warten (Kunde 0 wird fertig), Kunde 3 kommt in Minute 3 und muss bis
Minute 6 warten (Kunde 1 wird fertig), also 3 Minuten. Kunde 4 kommt in Minute 10 und findet einen freien Schalter. Es gibt je Kunde ein Ankunfts- und ein Fertig-Ereignis, zusammen **10**.
Die Reihenfolge der Ereignisse bestimmt die Ereignisliste, nicht der Code: Ein Fertig-Ereignis wird erst dann gebraucht, wenn seine Zeit die kleinste in der Liste ist.

</details>

#### Frage 2.3

In `simuliere_schalter` ist `FERTIG = 0` und `ANKUNFT = 1`. Bei gleicher Zeit kommt also **zuerst** das Fertig-Ereignis.

1. Ändert sich die Liste der Wartezeiten, wenn du `FERTIG = 1` und `ANKUNFT = 0` setzt? Probiere es mit verschiedenen Kundenlisten aus (zum Beispiel `[(0, 5), (5, 3), (5, 2)]` mit einem Schalter).
2. Wann **kann** die Reihenfolge gleichzeitiger Ereignisse das Ergebnis ändern?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Wartezeiten ändern sich hier **nicht**: Nach **jedem** Ereignis startet die `while`-Schleife alle Wartenden, solange ein Schalter frei ist. Egal, ob erst die Ankunft und dann das Fertig dieser Minute
   verarbeitet wird oder umgekehrt, der Kunde bekommt den Schalter in dieser Minute (Wartezeit 0 bzw. dieselbe wie vorher).
2. Wenn eine Regel schon **beim einzelnen Ereignis** etwas entscheidet. Beispiel: „Kommt ein Kunde an und alle Schalter sind besetzt, geht er wieder.“ Dann hängt es davon ab, ob das Fertig dieser Minute schon
   verarbeitet wurde. Bei Aufgaben, in denen zwei Dinge gleichzeitig geschehen, legt der Aufgabentext oder deine Annahme fest, was zuerst kommt. Wichtig ist:
   **Lege die Reihenfolge bewusst fest** (hier durch die Zahlen 0 und 1 im Tupel) und schreibe sie in die Dokumentation.

</details>

### Kennzahlen: mittlere und größte Wartezeit

Meist fragt die Aufgabe nach **Kennzahlen**, nicht nach der ganzen Liste: der **mittleren** Wartezeit (Summe geteilt durch Anzahl) und der **größten**. Für Tests brauchst du viele Kunden. Die erzeugen wir mit
einem **festen Startwert**, damit bei dir dieselben Kunden herauskommen wie hier (mehr dazu in Abschnitt 4). Die Abstände der Ankünfte liegen zwischen 1 und 12 Minuten, die Bedienzeiten zwischen 3 und 14.

#### Aufgabe 2.4

`mach_kunden(startwert, anzahl)` und `simuliere_schalter` sind fertig. Schreibe `kennzahlen(wartezeiten)`: Sie gibt `(mittel, groesste)` zurück. Die Ausgabe vergleicht
1, 2 und 3 Schalter bei denselben 40 Kunden.

**Erwartete Ausgabe:**
```
1 Schalter: mittlere Wartezeit 40.6, größte 72
2 Schalter: mittlere Wartezeit 0.9, größte 7
3 Schalter: mittlere Wartezeit 0.0, größte 1
```

In [ ]:
import random

def mach_kunden(startwert, anzahl):
    zufall = random.Random(startwert)          # eigener Zufallsgenerator mit festem Startwert
    kunden = []
    zeit = 0
    for i in range(anzahl):
        zeit = zeit + zufall.randint(1, 12)    # Abstand zum vorigen Kunden
        kunden.append((zeit, zufall.randint(3, 14)))
    return kunden

from heapq import heappush, heappop

FERTIG = 0
ANKUNFT = 1

def simuliere_schalter(kunden, anzahl):
    ereignisliste = []
    for nummer in range(len(kunden)):
        heappush(ereignisliste, (kunden[nummer][0], ANKUNFT, nummer))
    frei = anzahl
    wartend = []
    wartezeiten = [0] * len(kunden)
    ereignisse = 0
    while len(ereignisliste) > 0:
        zeit, art, nummer = heappop(ereignisliste)
        ereignisse = ereignisse + 1
        if art == ANKUNFT:
            wartend.append(nummer)
        else:
            frei = frei + 1
        while frei > 0 and len(wartend) > 0:
            naechster = wartend.pop(0)
            frei = frei - 1
            wartezeiten[naechster] = zeit - kunden[naechster][0]
            heappush(ereignisliste, (zeit + kunden[naechster][1], FERTIG, naechster))
    return wartezeiten, ereignisse

def kennzahlen(wartezeiten):
    # dein Code
    return 0, 0

kunden = mach_kunden(7, 40)
for anzahl in [1, 2, 3]:
    wartezeiten, ereignisse = simuliere_schalter(kunden, anzahl)
    mittel, groesste = kennzahlen(wartezeiten)
    print(anzahl, 'Schalter: mittlere Wartezeit %.1f, größte %d' % (mittel, groesste))

<details>
<summary>Tipp anzeigen</summary>

`sum(liste)` addiert alle Zahlen einer Liste, `max(liste)` liefert die größte, `len(liste)` die Anzahl.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random

def mach_kunden(startwert, anzahl):
    zufall = random.Random(startwert)          # eigener Zufallsgenerator mit festem Startwert
    kunden = []
    zeit = 0
    for i in range(anzahl):
        zeit = zeit + zufall.randint(1, 12)    # Abstand zum vorigen Kunden
        kunden.append((zeit, zufall.randint(3, 14)))
    return kunden

from heapq import heappush, heappop

FERTIG = 0
ANKUNFT = 1

def simuliere_schalter(kunden, anzahl):
    ereignisliste = []
    for nummer in range(len(kunden)):
        heappush(ereignisliste, (kunden[nummer][0], ANKUNFT, nummer))
    frei = anzahl
    wartend = []
    wartezeiten = [0] * len(kunden)
    ereignisse = 0
    while len(ereignisliste) > 0:
        zeit, art, nummer = heappop(ereignisliste)
        ereignisse = ereignisse + 1
        if art == ANKUNFT:
            wartend.append(nummer)
        else:
            frei = frei + 1
        while frei > 0 and len(wartend) > 0:
            naechster = wartend.pop(0)
            frei = frei - 1
            wartezeiten[naechster] = zeit - kunden[naechster][0]
            heappush(ereignisliste, (zeit + kunden[naechster][1], FERTIG, naechster))
    return wartezeiten, ereignisse

def kennzahlen(wartezeiten):
    mittel = sum(wartezeiten) / len(wartezeiten)
    groesste = max(wartezeiten)
    return mittel, groesste

kunden = mach_kunden(7, 40)
for anzahl in [1, 2, 3]:
    wartezeiten, ereignisse = simuliere_schalter(kunden, anzahl)
    mittel, groesste = kennzahlen(wartezeiten)
    print(anzahl, 'Schalter: mittlere Wartezeit %.1f, größte %d' % (mittel, groesste))
```

Mit **einem** Schalter wird die Schlange immer länger (die Bedienzeit ist im Mittel 8,5 Minuten, der Abstand der Kunden im Mittel nur 6,5): Die mittlere Wartezeit liegt bei rund 41 Minuten, die größte bei 72.
Ein **zweiter** Schalter bringt sie auf unter eine Minute (größte Wartezeit 7), mit einem dritten wartet fast niemand (größte Wartezeit 1). Das ist typisch: Sobald die Bedienung **schneller** ist als die Ankunft, verschwindet die lange Schlange schlagartig.

</details>

## 3. Finde den Fehler: Ereignisse nicht der Zeit nach

Ein Mitschüler hat die Simulation für **einen** Schalter ohne `heapq` geschrieben. Die Ankünfte stehen in einer ganz normalen Liste, mit `pop(0)` nimmt er das **erste** Ereignis.
Die neu entstehenden Fertig-Ereignisse hängt er **hinten** an (`append`).

#### Aufgabe 3.1

Führe die Zelle aus. Im ersten Beispiel kommt Kunde 1 erst in Minute 10 an, Kunde 0 ist aber schon in Minute 5 fertig. Was fällt dir bei den Ergebnissen auf? Warum passiert das?
Repariere die Funktion, indem du `append` und `pop(0)` durch `heappush` und `heappop` ersetzt (und die Liste zu Beginn mit `heapify`, oder gleich mit `heappush` füllst). Die Ausgabe ist die der **richtigen** Funktion.

**Erwartete Ausgabe:**
```
[0, 0]
[0, 4, 8]
```

In [ ]:
def simuliere_falsch(kunden):
    ereignisliste = []
    for nummer in range(len(kunden)):
        ereignisliste.append((kunden[nummer][0], 1, nummer))       # alle Ankünfte
    frei = 1
    wartend = []
    wartezeiten = [0] * len(kunden)
    while len(ereignisliste) > 0:
        zeit, art, nummer = ereignisliste.pop(0)                    # das erste Ereignis der Liste
        if art == 1:
            wartend.append(nummer)
        else:
            frei = frei + 1
        while frei > 0 and len(wartend) > 0:
            naechster = wartend.pop(0)
            frei = frei - 1
            wartezeiten[naechster] = zeit - kunden[naechster][0]
            ereignisliste.append((zeit + kunden[naechster][1], 0, naechster))   # hinten anhängen
    return wartezeiten

print(simuliere_falsch([(0, 5), (10, 2)]))
print(simuliere_falsch([(0, 5), (1, 5), (2, 5)]))

<details>
<summary>Tipp anzeigen</summary>

Das Fertig-Ereignis von Kunde 0 steht in der Liste **hinter** der Ankunft von Kunde 1 (Minute 10), obwohl es mit Minute 5 früher liegt. Es wird zu spät verarbeitet.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from heapq import heappush, heappop

def simuliere_falsch(kunden):
    ereignisliste = []
    for nummer in range(len(kunden)):
        heappush(ereignisliste, (kunden[nummer][0], 1, nummer))
    frei = 1
    wartend = []
    wartezeiten = [0] * len(kunden)
    while len(ereignisliste) > 0:
        zeit, art, nummer = heappop(ereignisliste)                  # das Ereignis mit der kleinsten Zeit
        if art == 1:
            wartend.append(nummer)
        else:
            frei = frei + 1
        while frei > 0 and len(wartend) > 0:
            naechster = wartend.pop(0)
            frei = frei - 1
            wartezeiten[naechster] = zeit - kunden[naechster][0]
            heappush(ereignisliste, (zeit + kunden[naechster][1], 0, naechster))
    return wartezeiten

print(simuliere_falsch([(0, 5), (10, 2)]))
print(simuliere_falsch([(0, 5), (1, 5), (2, 5)]))
```

Der Fehler: Die Liste ist **nicht nach der Zeit** geordnet. Die Ankunft in Minute 10 wird bearbeitet, bevor das Fertig von Minute 5 an der Reihe war: Der Schalter erscheint besetzt,
Kunde 1 reiht sich ein und beginnt erst, wenn das Fertig-Ereignis verarbeitet wird, also (mit der Zeit 5) **vor** seiner Ankunft. Das ergibt die Wartezeit **-5**: Eine **negative Wartezeit** ist immer ein
Hinweis auf einen Fehler (daran erkennst du ihn auch ohne Vergleichswerte). Mit `heappush` und `heappop` werden die Ereignisse **immer** der Zeit nach bearbeitet, auch die später eingefügten.

</details>

## 4. Zufall mit festem Startwert und Monte-Carlo

Manche Fragen kann man nicht ausrechnen, aber man kann sie **ausprobieren**: Man spielt das Zufallsexperiment **sehr oft** nach und zählt, wie oft das Ereignis eintritt. Der Anteil nähert sich
der **Wahrscheinlichkeit**. Das heißt **Monte-Carlo-Verfahren** (nach dem Spielkasino).

**Zufallszahlen am Computer** sind eigentlich **berechnet**: Aus einem **Startwert** wird eine Folge von Zahlen erzeugt, die zufällig aussieht. Mit `random.seed(zahl)` legst du den Startwert fest, danach liefert
`random.random()` immer wieder eine Zahl zwischen 0 und 1 (jede ist gleich wahrscheinlich) und `random.randint(a, b)` eine ganze Zahl von `a` bis `b`. Setzt du denselben Startwert, kommt **dieselbe Folge** heraus.

#### Aufgabe 4.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
import random

random.seed(5)
a = [random.randint(1, 6) for i in range(5)]
random.seed(5)
b = [random.randint(1, 6) for i in range(5)]
random.seed(6)
c = [random.randint(1, 6) for i in range(5)]
print(a == b)
print(len(c))
print(a == c)

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
True
5
False
```

Beide Male wird mit dem Startwert 5 begonnen, also kommt dieselbe Folge heraus (`True`). Die Liste `c` hat 5 Würfe (`5`). Mit dem Startwert 6 kommt eine andere Folge heraus, daher ist `a == c`
fast sicher `False`. Welche Zahlen genau, ist hier nicht wichtig, nur dass sie bei **gleichem Startwert** gleich sind.

</details>

**Warum ist das für den Wettbewerb wichtig?** Wer in der Dokumentation schreibt „Der beste Spieler gewinnt in 35,6 % der Turniere“, aber dann die Zahl bei jedem Programmstart ein bisschen anders ausgibt,
kann seine Ergebnisse niemandem zeigen. Mit **festem Startwert** (den du **in der Dokumentation nennst**) kann jeder dein Programm laufen lassen und bekommt **genau deine Zahlen**.
Die Zufälligkeit bleibt trotzdem: Mit einem anderen Startwert bekommst du eine andere, ebenso gute Schätzung.

*Hinweis zu diesem Notebook:* Die Zufallsfolge zu einem Startwert ist in Python für `random()` und `randint` festgelegt, bei dir kommen also dieselben Zahlen heraus wie hier. Trotzdem geben die Aufgaben
nur Werte **mit Spielraum** an, damit eine kleine Abweichung nicht stört.

### Das Beispiel: ein Turnier mit vier Spielern

Vier Spieler haben die Spielstärken `[20, 30, 40, 100]` (Spieler 0 bis 3). Bei einem **Spiel** zwischen zwei Spielern mit den Stärken `a` und `b` gewinnt der erste mit der Wahrscheinlichkeit
`a / (a + b)` (denk an eine Urne mit `a` roten und `b` blauen Kugeln). Im **K.o.-System** mit Plan `[0, 1, 2, 3]` spielen zuerst 0 gegen 1 und 2 gegen 3, die Sieger spielen im Finale. **Wie wahrscheinlich
gewinnt der stärkste Spieler** (hier Spieler 3)?

#### Aufgabe 4.2

Schreibe `spiel(staerke, x, y)`: `True`, wenn Spieler `x` gewinnt (mit der Wahrscheinlichkeit `staerke[x] / (staerke[x] + staerke[y])`, also `random.random() < …`), sonst `False`.
Schreibe dann `ko(staerke, plan)`: Der Plan ist eine Liste der Spieler, je zwei aufeinanderfolgende spielen gegeneinander. Die Sieger bilden den **neuen Plan**, das wiederholt sich, bis nur einer übrig ist
(Rückgabe: dessen Nummer). Der Testaufruf lässt 1000 Turniere spielen. Die Ausgabe zeigt die ersten drei Spiele, ob der stärkste Spieler mindestens 400 Mal gewonnen hat, und
welche Spieler überhaupt gewonnen haben.

**Erwartete Ausgabe:**
```
True False False
True
[0, 1, 2, 3]
```

In [ ]:
import random

def spiel(staerke, x, y):
    # dein Code
    return True

def ko(staerke, plan):
    # dein Code
    return plan[0]

staerke = [20, 30, 40, 100]
random.seed(1)
print(spiel(staerke, 0, 1), spiel(staerke, 0, 1), spiel(staerke, 0, 1))
gewinner = []
for i in range(1000):
    gewinner.append(ko(staerke, [0, 1, 2, 3]))
print(gewinner.count(3) >= 400)
print(sorted(set(gewinner)))

<details>
<summary>Tipp anzeigen</summary>

`random.random() < a / (a + b)` ist mit der Wahrscheinlichkeit `a / (a + b)` wahr. Bei `ko`: Eine `while`-Schleife, solange der Plan mehr als einen Spieler hat; `range(0, len(plan), 2)` zählt in Zweierschritten.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def ko(staerke, plan):
    while len(plan) > 1:
        neu = []
        for i in range(0, len(plan), 2):                 # die Paare: (0, 1), (2, 3), ...
            if spiel(staerke, plan[i], plan[i + 1]):
                neu.append(plan[i])
            else:
                neu.append(plan[i + 1])
        plan = neu
    return plan[0]

staerke = [20, 30, 40, 100]
random.seed(1)
print(spiel(staerke, 0, 1), spiel(staerke, 0, 1), spiel(staerke, 0, 1))
gewinner = []
for i in range(1000):
    gewinner.append(ko(staerke, [0, 1, 2, 3]))
print(gewinner.count(3) >= 400)
print(sorted(set(gewinner)))
```

Das Zufallsergebnis der ersten drei Spiele ist beliebig (aber mit dem Startwert 1 immer gleich). Der Stärkste (Spieler 3) gewinnt mindestens 400 von 1000 Turnieren (`True`), und es gewinnt nicht immer
er allein: Auch die schwächeren Spieler können gewinnen (die Liste enthält alle vier).

</details>

#### Aufgabe 4.3

`spiel` und `ko` sind fertig. Schreibe `schaetze(staerke, plan, wiederholungen, startwert)`: Sie setzt **einmal** `random.seed(startwert)`, lässt `wiederholungen` Turniere spielen und gibt den **Anteil**
zurück, in dem der Spieler mit der größten Spielstärke gewonnen hat (`staerke.index(max(staerke))` ist seine Nummer).

**Erwartete Ausgabe:**
```
100 0.57
1000 0.565
10000 0.5659
```

In [ ]:
import random

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def ko(staerke, plan):
    while len(plan) > 1:
        neu = []
        for i in range(0, len(plan), 2):
            if spiel(staerke, plan[i], plan[i + 1]):
                neu.append(plan[i])
            else:
                neu.append(plan[i + 1])
        plan = neu
    return plan[0]

def schaetze(staerke, plan, wiederholungen, startwert):
    # dein Code
    return 0

staerke = [20, 30, 40, 100]
for wiederholungen in [100, 1000, 10000]:
    print(wiederholungen, schaetze(staerke, [0, 1, 2, 3], wiederholungen, 1))

<details>
<summary>Tipp anzeigen</summary>

Zähle in einer Variablen `treffer`, wie oft `ko(staerke, plan)` gleich der Nummer des Stärksten ist, und teile am Ende durch `wiederholungen`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def ko(staerke, plan):
    while len(plan) > 1:
        neu = []
        for i in range(0, len(plan), 2):
            if spiel(staerke, plan[i], plan[i + 1]):
                neu.append(plan[i])
            else:
                neu.append(plan[i + 1])
        plan = neu
    return plan[0]

def schaetze(staerke, plan, wiederholungen, startwert):
    random.seed(startwert)                               # EINMAL vor allen Wiederholungen
    stärkster = staerke.index(max(staerke))
    treffer = 0
    for i in range(wiederholungen):
        if ko(staerke, plan) == stärkster:
            treffer = treffer + 1
    return treffer / wiederholungen

staerke = [20, 30, 40, 100]
for wiederholungen in [100, 1000, 10000]:
    print(wiederholungen, schaetze(staerke, [0, 1, 2, 3], wiederholungen, 1))
```

Die drei Zahlen (0,57, 0,565 und 0,566) liegen nahe beieinander. Das ist bei **diesem** Startwert so: Mit **100** Wiederholungen ist die Schätzung eigentlich noch grob und kann bei einem anderen Startwert viel weiter
danebenliegen. Wie weit, siehst du in Aufgabe 4.5.

</details>

### Die genaue Wahrscheinlichkeit: alle Fälle zählen

Bei **vier** Spielern gibt es nur **drei** Spiele, jedes geht auf zwei Arten aus: Das sind `2 · 2 · 2 = 8` Möglichkeiten. Wie in [Notebook 1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/01_durchprobieren.ipynb) kann man **alle
Fälle durchgehen** und jeden mit seiner Wahrscheinlichkeit gewichten: Das Produkt der Wahrscheinlichkeiten der drei Spielausgänge. Man addiert die Wahrscheinlichkeiten der Fälle, in denen der Stärkste
gewinnt. `itertools.product([True, False], repeat=3)` liefert alle acht Ausgänge (`True`: der Erste im Spiel gewinnt). Die Funktion ist **fertig**, lies sie in Ruhe:

In [ ]:
import itertools

def exakt(staerke, plan):
    """Genaue Wahrscheinlichkeit, dass der Stärkste das K.o.-Turnier gewinnt (alle Spielausgänge durchgehen)."""
    stärkster = staerke.index(max(staerke))
    summe = 0
    for ausgaenge in itertools.product([True, False], repeat=len(plan) - 1):    # ein Ausgang je Spiel
        w = 1                                    # Wahrscheinlichkeit dieses Falls
        runde = plan
        k = 0                                    # Nummer des Spiels
        while len(runde) > 1:
            neu = []
            for i in range(0, len(runde), 2):
                x = runde[i]
                y = runde[i + 1]
                p = staerke[x] / (staerke[x] + staerke[y])      # x gewinnt mit dieser Wahrscheinlichkeit
                if ausgaenge[k]:
                    w = w * p
                    neu.append(x)
                else:
                    w = w * (1 - p)
                    neu.append(y)
                k = k + 1
            runde = neu
        if runde[0] == stärkster:
            summe = summe + w
    return summe

print(round(exakt([20, 30, 40, 100], [0, 1, 2, 3]), 4))

#### Frage 4.4

1. Der stärkste Spieler gewinnt also mit genau **56,78 %**. Passt das zu den Schätzungen aus 4.3?
2. Wie viele Fälle muss `exakt` bei 8 Spielern durchgehen, wie viele bei 16, wie viele bei 64? Warum nimmt man bei großen Turnieren trotzdem das Monte-Carlo-Verfahren?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Ja: Die Schätzungen lagen bei etwa 0,56 bis 0,57, die Abweichung wird mit mehr Wiederholungen kleiner.
2. Bei `n` Spielern gibt es `n - 1` Spiele, also `2^(n-1)` Fälle: **128** bei 8 Spielern, **32 768** bei 16 Spielern und bei 64 Spielern 2^63, rund 9 · 10^18, also unmöglich. Die Simulation braucht dagegen
   nur **Wiederholungen · (n − 1)** Spiele. Ihr Preis: Das Ergebnis ist nur **geschätzt**. Wo der genaue Wert zu bekommen ist, dient er als **Kontrolle** der Simulation (wie hier).

</details>

#### Aufgabe 4.5

Eine Schätzung hängt vom **Startwert** ab. Wie stark? Schreibe `streuung(staerke, plan, wiederholungen)`: Sie ruft `schaetze` mit den **zehn Startwerten** `0` bis `9` auf und gibt die **kleinste**
und die **größte** der zehn Schätzungen als Paar zurück. (`schaetze`, `ko` und `exakt` sind fertig.)

**Erwartete Ausgabe:**
```
genau: 0.5678
  100 Wiederholungen: von 0.450 bis 0.680
 1000 Wiederholungen: von 0.527 bis 0.575
10000 Wiederholungen: von 0.558 bis 0.578
```

In [ ]:
import random

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def ko(staerke, plan):
    while len(plan) > 1:
        neu = []
        for i in range(0, len(plan), 2):
            if spiel(staerke, plan[i], plan[i + 1]):
                neu.append(plan[i])
            else:
                neu.append(plan[i + 1])
        plan = neu
    return plan[0]

def schaetze(staerke, plan, wiederholungen, startwert):
    random.seed(startwert)                               # EINMAL vor allen Wiederholungen
    stärkster = staerke.index(max(staerke))
    treffer = 0
    for i in range(wiederholungen):
        if ko(staerke, plan) == stärkster:
            treffer = treffer + 1
    return treffer / wiederholungen

import itertools

def exakt(staerke, plan):
    """Genaue Wahrscheinlichkeit, dass der Stärkste das K.o.-Turnier gewinnt (alle Spielausgänge durchgehen)."""
    stärkster = staerke.index(max(staerke))
    summe = 0
    for ausgaenge in itertools.product([True, False], repeat=len(plan) - 1):    # ein Ausgang je Spiel
        w = 1                                    # Wahrscheinlichkeit dieses Falls
        runde = plan
        k = 0                                    # Nummer des Spiels
        while len(runde) > 1:
            neu = []
            for i in range(0, len(runde), 2):
                x = runde[i]
                y = runde[i + 1]
                p = staerke[x] / (staerke[x] + staerke[y])      # x gewinnt mit dieser Wahrscheinlichkeit
                if ausgaenge[k]:
                    w = w * p
                    neu.append(x)
                else:
                    w = w * (1 - p)
                    neu.append(y)
                k = k + 1
            runde = neu
        if runde[0] == stärkster:
            summe = summe + w
    return summe


def streuung(staerke, plan, wiederholungen):
    # dein Code
    return 0, 0

staerke = [20, 30, 40, 100]
print('genau: %.4f' % exakt(staerke, [0, 1, 2, 3]))
for wiederholungen in [100, 1000, 10000]:
    kleinste, groesste = streuung(staerke, [0, 1, 2, 3], wiederholungen)
    print('%5d Wiederholungen: von %.3f bis %.3f' % (wiederholungen, kleinste, groesste))

<details>
<summary>Tipp anzeigen</summary>

Eine Liste `schaetzungen`, in einer Schleife `for startwert in range(10):` das Ergebnis von `schaetze(...)` anhängen. Am Ende `min` und `max`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def ko(staerke, plan):
    while len(plan) > 1:
        neu = []
        for i in range(0, len(plan), 2):
            if spiel(staerke, plan[i], plan[i + 1]):
                neu.append(plan[i])
            else:
                neu.append(plan[i + 1])
        plan = neu
    return plan[0]

def schaetze(staerke, plan, wiederholungen, startwert):
    random.seed(startwert)                               # EINMAL vor allen Wiederholungen
    stärkster = staerke.index(max(staerke))
    treffer = 0
    for i in range(wiederholungen):
        if ko(staerke, plan) == stärkster:
            treffer = treffer + 1
    return treffer / wiederholungen

import itertools

def exakt(staerke, plan):
    """Genaue Wahrscheinlichkeit, dass der Stärkste das K.o.-Turnier gewinnt (alle Spielausgänge durchgehen)."""
    stärkster = staerke.index(max(staerke))
    summe = 0
    for ausgaenge in itertools.product([True, False], repeat=len(plan) - 1):    # ein Ausgang je Spiel
        w = 1                                    # Wahrscheinlichkeit dieses Falls
        runde = plan
        k = 0                                    # Nummer des Spiels
        while len(runde) > 1:
            neu = []
            for i in range(0, len(runde), 2):
                x = runde[i]
                y = runde[i + 1]
                p = staerke[x] / (staerke[x] + staerke[y])      # x gewinnt mit dieser Wahrscheinlichkeit
                if ausgaenge[k]:
                    w = w * p
                    neu.append(x)
                else:
                    w = w * (1 - p)
                    neu.append(y)
                k = k + 1
            runde = neu
        if runde[0] == stärkster:
            summe = summe + w
    return summe


def streuung(staerke, plan, wiederholungen):
    schaetzungen = []
    for startwert in range(10):
        schaetzungen.append(schaetze(staerke, plan, wiederholungen, startwert))
    return min(schaetzungen), max(schaetzungen)

staerke = [20, 30, 40, 100]
print('genau: %.4f' % exakt(staerke, [0, 1, 2, 3]))
for wiederholungen in [100, 1000, 10000]:
    kleinste, groesste = streuung(staerke, [0, 1, 2, 3], wiederholungen)
    print('%5d Wiederholungen: von %.3f bis %.3f' % (wiederholungen, kleinste, groesste))
```

Mit **100** Wiederholungen liegen die zehn Schätzungen zwischen 0,45 und 0,68, also **weit** vom genauen Wert 0,5678 entfernt. Mit **1000** schrumpft die Spanne auf etwa 0,53 bis 0,58, mit **10 000**
auf etwa 0,56 bis 0,58. Alle Schätzungen mit 10 000 Wiederholungen liegen höchstens etwa 0,01 vom genauen Wert entfernt.

</details>

#### Frage 4.6

Wie viele Wiederholungen braucht man, um die Schätzung **zehnmal genauer** zu machen? Rechne dazu mit der Faustregel: Der typische Fehler einer Schätzung mit `N` Wiederholungen ist etwa `0,5 / Wurzel(N)`.
Vergleiche mit der Tabelle aus 4.5.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Bei `N = 100` ist der typische Fehler etwa `0,5 / 10 = 0,05`, bei `N = 10 000` etwa `0,5 / 100 = 0,005`: **hundertmal** mehr Wiederholungen für **zehnmal** mehr Genauigkeit. Die Tabelle passt dazu: Bei
`N = 100` weichen einzelne Schätzungen um 0,1 und mehr ab, bei `N = 10 000` um etwa 0,01. Für eine Schätzung auf
**zwei Nachkommastellen** (±0,005) braucht man also etwa 10 000 Wiederholungen. Mehr Stellen anzugeben, wäre **vorgetäuschte Genauigkeit**: Gib in der Dokumentation nur so viele Stellen an, wie die
Wiederholungen hergeben.

</details>

#### Aufgabe 4.7

Ein Schüler will die Wahrscheinlichkeit schätzen, dass Spieler 3 (Stärke 100) gegen Spieler 0 (Stärke 20) gewinnt. Genau sind das `100 / 120 = 0,833`. Seine Schätzungen sind dagegen **seltsam**.
Was fällt dir an der Ausgabe auf, und wo liegt der Fehler? Repariere `ein_turnier` und `schaetze_falsch` so, dass der Startwert **nur einmal** gesetzt wird. Die Ausgabe ist die der reparierten Funktion;
die Werte sind Schätzungen, nur die Größenordnung (nahe bei 0,83) ist wichtig.

**Erwartete Ausgabe:**
```
0.9
0.822
```

In [ ]:
import random

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def ein_turnier(startwert):
    random.seed(startwert)                        # hier wird der Startwert gesetzt ...
    return spiel([20, 30, 40, 100], 3, 0)          # ... und dann ein Spiel gespielt (Spieler 3 gegen 0)

def schaetze_falsch(wiederholungen):
    treffer = 0
    for i in range(wiederholungen):
        if ein_turnier(42):                       # ... jedes Mal mit demselben Startwert
            treffer = treffer + 1
    return treffer / wiederholungen

print(schaetze_falsch(10))
print(schaetze_falsch(1000))

<details>
<summary>Tipp anzeigen</summary>

`random.seed(42)` startet die Zahlenfolge **von vorn**. Was liefert dann `random.random()` bei jedem Aufruf?

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def ein_turnier():
    return spiel([20, 30, 40, 100], 3, 0)

def schaetze_falsch(wiederholungen):
    random.seed(42)                               # EINMAL, vor den Wiederholungen
    treffer = 0
    for i in range(wiederholungen):
        if ein_turnier():
            treffer = treffer + 1
    return treffer / wiederholungen

print(schaetze_falsch(10))
print(schaetze_falsch(1000))
```

Im fehlerhaften Programm startet **jede** Wiederholung mit `random.seed(42)`, also kommt in jeder Wiederholung **dieselbe** Zufallszahl heraus: Alle Wiederholungen haben dasselbe Ergebnis, und die Schätzung ist
entweder **0,0** oder **1,0** (und für 10 und 1000 Wiederholungen **gleich**). Das ist keine Schätzung mehr. Richtig ist: `random.seed(...)` **einmal vor** den Wiederholungen. Danach liefert jeder Aufruf die
**nächste** Zahl der Folge, und die Wiederholungen unterscheiden sich.

</details>

## Wettbewerbs-Ecke 1: Fahrradwerkstatt

*Angelehnt an Aufgabe 4 „Fahrradwerkstatt“ der 1. Runde des 41. Bundeswettbewerbs Informatik (2022/23).*

Marc repariert Fahrräder. Für jeden **Auftrag** ist der **Eingangszeitpunkt** (Minuten seit dem Beginn von Tag 0 um 0 Uhr) und die **Bearbeitungsdauer** (Minuten) bekannt. Marc arbeitet nur
**von 9 bis 17 Uhr**, ein Auftrag kann über Nacht liegen bleiben. Er arbeitet immer nur an **einem** Auftrag. Zwei Verfahren:

- **A:** in der Reihenfolge des Eingangs,
- **B:** immer den **kürzesten** der vorliegenden Aufträge.

Die Datei `daten/fahrradwerkstattN.txt` (N = 0 bis 4) hat in jeder Zeile einen Auftrag: Eingang und Dauer. Die Aufgabe wird hier **vereinfacht**: Die **Nacht rechnen wir weg**. Die Funktion `arbeitszeit(t)` im
Gerüst wandelt einen Zeitpunkt in **Arbeitsminuten** um, also die Zeit, in der Marc überhaupt arbeitet (`tag * 480 +` der Teil des Tages zwischen 9 und 17 Uhr). Du simulierst Marc dann nur auf dieser
Arbeitsuhr und gibst die mittlere und die größte **Wartezeit** (Fertigstellung minus Eingang) **in Arbeitsminuten** aus. Die Aufträge **warten** in einer Prioritätswarteschlange: bei A nach Reihenfolge des Eingangs, bei B
nach der Dauer.

#### Aufgabe E.1

Schreibe `werkstatt(auftraege, verfahren)` mit `verfahren` gleich `'A'` oder `'B'`. Sie gibt `(mittel, groesste)` der Wartezeiten in Arbeitsminuten zurück. Gehe so vor:

1. Rechne alle Eingänge mit `arbeitszeit` um und sortiere die Aufträge nach Eingang: `eingaenge = sorted((arbeitszeit(e), d) for e, d in auftraege)`.
2. Die Arbeitsuhr `T` beginnt bei 0. In `wartend` (ein `heapq`) liegen die schon **eingegangenen** Aufträge. Der Eintrag ist bei **B** `(dauer, nummer, eingang)` und bei **A** `(nummer, dauer, eingang)`;
   die `nummer` (Stelle in der sortierten Liste) hält die Reihenfolge bei gleicher Dauer fest.
3. Wiederhole, bis alle Aufträge fertig sind: Alle Aufträge mit `eingang <= T` kommen in `wartend`. Ist `wartend` leer, **springe** `T` auf den nächsten Eingang. Sonst nimm den obersten Eintrag,
   `T` wächst um die Dauer, und die Wartezeit `T - eingang` merkst du dir.

**Erwartete Ausgabe:**
```
Datei  Aufträge |  A: Mittel   Max |  B: Mittel    Max
    0      114 |     11000 22943 |      5764  62974
    1      766 |     21278 42881 |      4059 144603
    2      186 |     17155 37053 |      5051 109167
    3      220 |     10112 20501 |      5836 127616
    4       90 |     24913 55699 |     14169 121235
```

In [ ]:
from heapq import heappush, heappop

def lies_auftraege(dateiname):
    auftraege = []
    with open(dateiname, encoding='utf-8') as datei:
        for zeile in datei:
            if zeile.strip():
                eingang, dauer = zeile.split()
                auftraege.append((int(eingang), int(dauer)))
    return auftraege

def arbeitszeit(t):
    """Arbeitsminuten bis zum Zeitpunkt t (Marc arbeitet täglich von Minute 540 bis 1020)."""
    tag, rest = divmod(t, 24 * 60)
    return tag * 480 + min(max(rest - 540, 0), 480)

def werkstatt(auftraege, verfahren):
    # dein Code
    return 0, 0

print('Datei  Aufträge |  A: Mittel   Max |  B: Mittel    Max')
for nummer in range(5):
    auftraege = lies_auftraege('daten/fahrradwerkstatt%d.txt' % nummer)
    a = werkstatt(auftraege, 'A')
    b = werkstatt(auftraege, 'B')
    print('%5d %8d | %9.0f %5d | %9.0f %6d' % (nummer, len(auftraege), a[0], a[1], b[0], b[1]))

<details>
<summary>Tipp anzeigen</summary>

Zwei Schleifen: außen `while len(wartezeiten) < len(eingaenge):` (bis alle fertig sind), innen `while naechster < len(eingaenge) and eingaenge[naechster][0] <= T:` für die neuen Aufträge. Der Fall
„Werkstatt leer“ springt mit `T = eingaenge[naechster][0]` und `continue`. Bei **A** liegt die Nummer vorne im Tupel, bei **B** die Dauer.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from heapq import heappush, heappop

def lies_auftraege(dateiname):
    auftraege = []
    with open(dateiname, encoding='utf-8') as datei:
        for zeile in datei:
            if zeile.strip():
                eingang, dauer = zeile.split()
                auftraege.append((int(eingang), int(dauer)))
    return auftraege

def arbeitszeit(t):
    """Arbeitsminuten bis zum Zeitpunkt t (Marc arbeitet täglich von Minute 540 bis 1020)."""
    tag, rest = divmod(t, 24 * 60)
    return tag * 480 + min(max(rest - 540, 0), 480)

def werkstatt(auftraege, verfahren):
    eingaenge = sorted((arbeitszeit(e), d) for e, d in auftraege)
    wartend = []
    naechster = 0                                      # nächster Auftrag, der noch nicht eingegangen ist
    T = 0                                              # Arbeitsuhr
    wartezeiten = []
    while len(wartezeiten) < len(eingaenge):
        while naechster < len(eingaenge) and eingaenge[naechster][0] <= T:
            eingang, dauer = eingaenge[naechster]
            if verfahren == 'A':
                heappush(wartend, (naechster, dauer, eingang))
            else:
                heappush(wartend, (dauer, naechster, eingang))
            naechster = naechster + 1
        if len(wartend) == 0:
            T = eingaenge[naechster][0]                # Werkstatt leer: springe zum nächsten Eingang
            continue
        eintrag = heappop(wartend)
        if verfahren == 'A':
            dauer = eintrag[1]
        else:
            dauer = eintrag[0]
        T = T + dauer
        wartezeiten.append(T - eintrag[2])
    return sum(wartezeiten) / len(wartezeiten), max(wartezeiten)

print('Datei  Aufträge |  A: Mittel   Max |  B: Mittel    Max')
for nummer in range(5):
    auftraege = lies_auftraege('daten/fahrradwerkstatt%d.txt' % nummer)
    a = werkstatt(auftraege, 'A')
    b = werkstatt(auftraege, 'B')
    print('%5d %8d | %9.0f %5d | %9.0f %6d' % (nummer, len(auftraege), a[0], a[1], b[0], b[1]))
```

Bei **A** (Reihenfolge des Eingangs) wartet jeder Auftrag im Mittel lange (bei Datei 0 etwa 11 000 Arbeitsminuten, das sind rund 23 Arbeitstage), aber **keiner extrem** lange (größte 22 943). Bei **B** (kürzester zuerst)
ist die mittlere Wartezeit bei **allen** Dateien kleiner (zum Beispiel 5764 statt 11 000 bei Datei 0). Dafür ist die **größte** Wartezeit bei B deutlich größer (62 974 statt 22 943): Lange Aufträge werden
von immer neuen kurzen überholt, und das Warten **wächst**, solange Marc ausgelastet ist. Die Werte stimmen mit der Spalte „ohne Totzeit“ des Lösungsnotebooks zur echten Aufgabe überein.

</details>

#### Frage E.2

1. Welches Verfahren würdest du Marc empfehlen, wenn er die **Kunden im Durchschnitt** zufriedenstellen will, welches, wenn **kein Kunde zu lange** warten soll?
2. Warum hilft eine Prioritätswarteschlange hier (statt bei jedem Auftrag die ganze Liste der Wartenden zu durchsuchen)? Wie viele Aufträge liegen bei Datei 1 im ungünstigsten Fall gleichzeitig vor?
3. Wie könnte man A und B **mischen**, damit lange Aufträge nicht ewig warten? (Eine Idee reicht.)

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Im Durchschnitt ist **B** besser (die mittlere Wartezeit ist bei allen Dateien um mindestens 40 % kleiner, bei Datei 1 sogar mehr als fünfmal), bei „niemand soll zu lange warten“ ist **A** besser (die größte Wartezeit ist bei A in allen Dateien kleiner).
2. Statt bei jedem Auftrag die ganze Liste der Wartenden zu durchsuchen (so viele Schritte, wie Aufträge warten), liefert `heappop` den kleinsten Eintrag mit sehr wenigen Schritten (etwa so viele wie
   die Zahl der Stellen von `n`, genauer `log n`). Bei Datei 1 liegen aber höchstens **36** Aufträge gleichzeitig vor: Hier würde auch die einfache Suche genügen. Der Haufen lohnt sich erst bei vielen tausend
   gleichzeitig wartenden Aufträgen. Er ist aber auch hier bequem, weil die Reihenfolge (nach Dauer oder nach Eingang) allein durch die **Tupelreihenfolge** festgelegt wird.
3. Zum Beispiel: „Kürzester zuerst, **aber** wartet ein Auftrag länger als eine Höchstgrenze, kommt er sofort dran.“ So behält man den kleinen Mittelwert von B und begrenzt das größte Warten. Das Lösungsnotebook
   zur echten Aufgabe rechnet das als Verfahren C durch.

</details>

## Wettbewerbs-Ecke 2: Tobis Turnier

*Angelehnt an Aufgabe 3 „Tobis Turnier“ der 1. Runde des 39. Bundeswettbewerbs Informatik (2020/21).*

Tobi organisiert ein Turnier für das Spiel RNG, bei dem nicht immer der Bessere gewinnt: Jeder Spieler hat eine **Spielstärke** von 0 bis 100. Bei einem Spiel gewinnt Spieler 1 mit der Wahrscheinlichkeit
`a / (a + b)` (wie oben). Tobi vergleicht zwei **K.o.-Pläne** und die Variante **K.o.×5**, bei der jedes Paar **fünfmal** spielt und der gewinnt, der **öfter** gewinnt. Gefragt: Wie oft gewinnt der
**stärkste** Spieler?

Die Dateien `daten/spielstaerkenN.txt` (N = 1 bis 4) enthalten in der ersten Zeile die Zahl der Spieler (8 oder 16) und dann je Zeile die Stärke eines Spielers. Du schätzt die Wahrscheinlichkeit mit
Monte-Carlo und prüfst sie mit der **genauen** Wahrscheinlichkeit (`exakt`, wie oben, nur für K.o. mit je einem Spiel, deshalb nur für die **8-Spieler-Dateien** 1 und 2).

#### Aufgabe E.3

Das Gerüst enthält `lies_staerken`, `spiel`, `ko(staerke, plan, serie)` (`serie` ist 1 oder 5), `exakt` und die zwei Pläne. Schreibe `schaetze(staerke, plan, serie, wiederholungen, startwert)`: Sie setzt
**einmal** den Startwert, spielt `wiederholungen` Turniere und gibt den **Anteil** zurück, in dem der Stärkste gewonnen hat. Die Ausgabe zeigt für die Dateien 1 und 2 die genaue Wahrscheinlichkeit
(`exakt`), die Schätzung mit je einem Spiel und die mit fünf Spielen.

**Erwartete Ausgabe:**
```
Datei 1 [0, 10, 20, 30, 40, 50, 60, 100]
  Plan A: genau 0.355, geschätzt 0.36, K.o.x5 geschätzt 0.56
  Plan B: genau 0.477, geschätzt 0.48, K.o.x5 geschätzt 0.60
Datei 2 [10, 10, 10, 10, 80, 80, 80, 100]
  Plan A: genau 0.281, geschätzt 0.29, K.o.x5 geschätzt 0.37
  Plan B: genau 0.307, geschätzt 0.30, K.o.x5 geschätzt 0.36
```

In [ ]:
import random
import itertools

def lies_staerken(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return zahlen[1:]                                   # zahlen[0] ist die Anzahl

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def paar(staerke, x, y, serie):
    """Wer kommt weiter? Das Paar spielt `serie` Mal, wer öfter gewinnt, kommt weiter."""
    siege = 0
    for i in range(serie):
        if spiel(staerke, x, y):
            siege = siege + 1
    if 2 * siege > serie:
        return x
    return y

def ko(staerke, plan, serie):
    while len(plan) > 1:
        neu = []
        for i in range(0, len(plan), 2):
            neu.append(paar(staerke, plan[i], plan[i + 1], serie))
        plan = neu
    return plan[0]

def exakt(staerke, plan):
    stärkster = staerke.index(max(staerke))
    summe = 0
    for ausgaenge in itertools.product([True, False], repeat=len(plan) - 1):
        w = 1
        runde = plan
        k = 0
        while len(runde) > 1:
            neu = []
            for i in range(0, len(runde), 2):
                x = runde[i]
                y = runde[i + 1]
                p = staerke[x] / (staerke[x] + staerke[y])
                if ausgaenge[k]:
                    w = w * p
                    neu.append(x)
                else:
                    w = w * (1 - p)
                    neu.append(y)
                k = k + 1
            runde = neu
        if runde[0] == stärkster:
            summe = summe + w
    return summe

PLAN_A = [0, 1, 2, 3, 4, 5, 6, 7]                       # 1 gegen 2, 3 gegen 4, ...
PLAN_B = [0, 7, 1, 6, 2, 5, 3, 4]                       # 1 gegen 8, 2 gegen 7, ...

def schaetze(staerke, plan, serie, wiederholungen, startwert):
    # dein Code
    return 0

for nummer in [1, 2]:
    staerke = lies_staerken('daten/spielstaerken%d.txt' % nummer)
    print('Datei', nummer, staerke)
    for name, plan in [('A', PLAN_A), ('B', PLAN_B)]:
        genau = exakt(staerke, plan)
        schaetzung = schaetze(staerke, plan, 1, 5000, 2020)
        fuenf = schaetze(staerke, plan, 5, 5000, 2020)
        print('  Plan %s: genau %.3f, geschätzt %.2f, K.o.x5 geschätzt %.2f' % (name, genau, schaetzung, fuenf))

<details>
<summary>Tipp anzeigen</summary>

Wie `schaetze` aus Aufgabe 4.3, nur dass `ko` jetzt noch den Parameter `serie` bekommt.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random
import itertools

def lies_staerken(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return zahlen[1:]                                   # zahlen[0] ist die Anzahl

def spiel(staerke, x, y):
    return random.random() < staerke[x] / (staerke[x] + staerke[y])

def paar(staerke, x, y, serie):
    """Wer kommt weiter? Das Paar spielt `serie` Mal, wer öfter gewinnt, kommt weiter."""
    siege = 0
    for i in range(serie):
        if spiel(staerke, x, y):
            siege = siege + 1
    if 2 * siege > serie:
        return x
    return y

def ko(staerke, plan, serie):
    while len(plan) > 1:
        neu = []
        for i in range(0, len(plan), 2):
            neu.append(paar(staerke, plan[i], plan[i + 1], serie))
        plan = neu
    return plan[0]

def exakt(staerke, plan):
    stärkster = staerke.index(max(staerke))
    summe = 0
    for ausgaenge in itertools.product([True, False], repeat=len(plan) - 1):
        w = 1
        runde = plan
        k = 0
        while len(runde) > 1:
            neu = []
            for i in range(0, len(runde), 2):
                x = runde[i]
                y = runde[i + 1]
                p = staerke[x] / (staerke[x] + staerke[y])
                if ausgaenge[k]:
                    w = w * p
                    neu.append(x)
                else:
                    w = w * (1 - p)
                    neu.append(y)
                k = k + 1
            runde = neu
        if runde[0] == stärkster:
            summe = summe + w
    return summe

PLAN_A = [0, 1, 2, 3, 4, 5, 6, 7]                       # 1 gegen 2, 3 gegen 4, ...
PLAN_B = [0, 7, 1, 6, 2, 5, 3, 4]                       # 1 gegen 8, 2 gegen 7, ...

def schaetze(staerke, plan, serie, wiederholungen, startwert):
    random.seed(startwert)
    stärkster = staerke.index(max(staerke))
    treffer = 0
    for i in range(wiederholungen):
        if ko(staerke, plan, serie) == stärkster:
            treffer = treffer + 1
    return treffer / wiederholungen

for nummer in [1, 2]:
    staerke = lies_staerken('daten/spielstaerken%d.txt' % nummer)
    print('Datei', nummer, staerke)
    for name, plan in [('A', PLAN_A), ('B', PLAN_B)]:
        genau = exakt(staerke, plan)
        schaetzung = schaetze(staerke, plan, 1, 5000, 2020)
        fuenf = schaetze(staerke, plan, 5, 5000, 2020)
        print('  Plan %s: genau %.3f, geschätzt %.2f, K.o.x5 geschätzt %.2f' % (name, genau, schaetzung, fuenf))
```

Die Schätzung mit je einem Spiel liegt nahe bei der genauen Wahrscheinlichkeit (bei Datei 1 Plan A: genau 0,355, geschätzt etwa 0,36). Plan B ist für den Stärksten
**besser** als Plan A (bei Datei 1 spielt er in der ersten Runde gegen Stärke 0 statt gegen Stärke 60), und noch viel besser ist **K.o.×5**: Bei Datei 1 steigt die Chance auf etwa 0,56 (Plan A) und 0,60 (Plan B). Je öfter ein Paar spielt, desto eher setzt sich der Stärkere durch.
Bei Datei 2 sind drei Spieler fast so stark wie der Beste, deshalb bleibt seine Chance auch bei K.o.×5 unter 0,4. Die Zahlen aus der Tabelle sind Schätzungen mit 5000 Wiederholungen: Rechne mit ±0,01.

</details>

#### Frage E.4

1. In den Dateien 3 und 4 spielen **16** Spieler. Wie viele Fälle müsste `exakt` für 16 Spieler durchgehen, und wie viele Spiele macht ein einziges K.o.-Turnier? Was folgt für die Wahl der Methode?
2. In Datei 4 hat ein Spieler die Stärke 100 und alle anderen 95. Würdest du erwarten, dass er **oft** gewinnt? Wie sähe die Schätzung ungefähr aus?
3. Was gehört in die **Dokumentation**, damit jemand deine Tabelle nachprüfen kann?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. 2^15 = **32 768** Fälle, jeder mit 15 Spielen (also rund eine halbe Million Spielausgänge): für eine Datei noch machbar, für 64 Spieler nicht mehr. Ein einzelnes Turnier hat nur 15 Spiele, auch 10 000 Turniere
   sind schnell berechnet. **Monte-Carlo** wächst nur mit der Zahl der Spiele, das genaue Zählen mit `2^(n-1)`. Bei großen Turnieren bleibt nur die Simulation.
2. Nein: Gegen jeden Spieler gewinnt er nur mit `100 / 195`, also etwa 51 %. Bei 16 Spielern und 4 Runden gewinnt er nur dann das Turnier, wenn er viermal hintereinander gewinnt: grob `0,51^4`, also **nur etwa 7 %**,
   und das ist ungefähr so viel wie bei jedem anderen Spieler (1/16 = 6,25 %). Die Stärke 100 gegen 95 ist praktisch **kein Vorteil**.
3. Die **Zahl der Wiederholungen**, den **Startwert**, die **Pläne** und die Beschreibung des **Urnenmodells**. Mit dem Startwert ist die Tabelle auf jedem Rechner **dieselbe**, und die Zahl der
   Wiederholungen sagt, wie viele Stellen sinnvoll sind (siehe Frage 4.6).

</details>

## Für Schnelle: Verzinkt, Kristalle wachsen im Raster

*Angelehnt an Aufgabe 2 „Verzinkt“ der 1. Runde des 41. Bundeswettbewerbs Informatik (2022/23).*

Auf feuerverzinktem Metall wachsen aus **Keimen** Kristalle, bis sie zusammenstoßen. Du simulierst das in einem kleinen **Raster** aus Zeichen. Jeder Keim hat einen Ort `(x, y)`, eine **Entstehungszeit** und
die **Dauer**, die er braucht, um ein Kästchen **waagerecht** (`dx`) bzw. **senkrecht** (`dy`) weiterzuwachsen (kleine Dauer = schnelles Wachstum). Ein Kästchen gehört dem Keim, der es **als Erster erreicht**.

**Idee:** Die **Ereignisse** sind „Keim `k` erreicht das Kästchen `(x, y)` zur Zeit `t`“. In einem `heapq` mit Einträgen `(zeit, nummer, x, y)` nimmt man immer das mit der kleinsten Zeit. Ist das Kästchen **frei**,
gehört es nun dem Keim, und die vier Nachbarn kommen mit der neuen Zeit (`t + dx` waagerecht, `t + dy` senkrecht) in die Liste. Ist es schon belegt, passiert nichts. **Bei gleicher Zeit entscheidet die kleinere Nummer**
des Keims. Das ist die Tupelreihenfolge, und sie legt den Gleichstand fest.

#### Aufgabe F.1

Schreibe `kristalle(breite, hoehe, keime)`. Sie gibt eine **Liste von Zeichenketten** (die Zeilen) zurück: `A`, `B`, … für die Kästchen des Keims 0, 1, … (nach der Nummer). Jeder Keim ist
ein Tupel `(x, y, start, dx, dy)`. Die Ausgabe zeigt zwei Keime in einem Raster von 10 mal 5 Kästchen und dann, wie viele Kästchen jeder Kristall hat.

**Erwartete Ausgabe:**
```
AAAAAAABBB
AAAAAAABBB
AAAAAABBBB
AAAAABBBBB
AAAAABBBBB
30 20
```

In [ ]:
from heapq import heappush, heappop

def kristalle(breite, hoehe, keime):
    belegt = [['.'] * breite for i in range(hoehe)]      # '.' = frei
    ereignisse = []
    for nummer in range(len(keime)):
        x, y, start, dx, dy = keime[nummer]
        heappush(ereignisse, (start, nummer, x, y))
    # dein Code
    return [''.join(zeile) for zeile in belegt]

keime = [(1, 1, 0, 1, 2), (8, 3, 0, 2, 1)]          # (x, y, Entstehungszeit, dx, dy)
bild = kristalle(10, 5, keime)
for zeile in bild:
    print(zeile)
alle = ''.join(bild)
print(alle.count('A'), alle.count('B'))

<details>
<summary>Tipp anzeigen</summary>

`while len(ereignisse) > 0:` mit `zeit, nummer, x, y = heappop(ereignisse)`. Ist `belegt[y][x] != '.'`, mache mit `continue` weiter. Sonst `belegt[y][x] = chr(ord('A') + nummer)` (`chr(ord('A') + 1)` ist `'B'`),
und für jeden der vier Nachbarn, der im Raster liegt und noch frei ist, `heappush(ereignisse, (zeit + dauer, nummer, nx, ny))`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
from heapq import heappush, heappop

def kristalle(breite, hoehe, keime):
    belegt = [['.'] * breite for i in range(hoehe)]
    ereignisse = []
    for nummer in range(len(keime)):
        x, y, start, dx, dy = keime[nummer]
        heappush(ereignisse, (start, nummer, x, y))
    while len(ereignisse) > 0:
        zeit, nummer, x, y = heappop(ereignisse)            # das früheste Ereignis
        if belegt[y][x] != '.':
            continue                                        # schon belegt: nichts tun
        belegt[y][x] = chr(ord('A') + nummer)              # das Kästchen gehört jetzt dem Keim
        dx = keime[nummer][3]
        dy = keime[nummer][4]
        for (nx, ny, dauer) in [(x - 1, y, dx), (x + 1, y, dx), (x, y - 1, dy), (x, y + 1, dy)]:
            if 0 <= nx < breite and 0 <= ny < hoehe and belegt[ny][nx] == '.':
                heappush(ereignisse, (zeit + dauer, nummer, nx, ny))
    return [''.join(zeile) for zeile in belegt]

keime = [(1, 1, 0, 1, 2), (8, 3, 0, 2, 1)]          # (x, y, Entstehungszeit, dx, dy)
bild = kristalle(10, 5, keime)
for zeile in bild:
    print(zeile)
alle = ''.join(bild)
print(alle.count('A'), alle.count('B'))
```

Der Keim A wächst **waagerecht** schnell (`dx = 1`) und senkrecht langsam (`dy = 2`), Keim B umgekehrt (`dx = 2`, `dy = 1`), deshalb sind die Kristalle langgezogen. Sie treffen sich in einer **Grenze** in der Mitte. Die
Kästchen auf der Grenze bekommt der, der sie zuerst erreicht. Bei **Gleichstand** der Zeit entscheidet die kleinere Nummer, die Tupelreihenfolge hat die Regel für gleichzeitige Ereignisse festgelegt.
Ein Kristall kann sich **nicht** in ein belegtes Kästchen hineinschieben, weil man belegte Kästchen überspringt.

</details>

## Das hast du gelernt

```python
from heapq import heappush, heappop

FERTIG = 0                                   # Art: bei gleicher Zeit zuerst
ANKUNFT = 1

ereignisse = []                              # Einträge (zeit, art, nummer)
heappush(ereignisse, (zeit, ANKUNFT, nummer))
while len(ereignisse) > 0:
    zeit, art, nummer = heappop(ereignisse)  # immer das früheste Ereignis
    ...                                      # reagieren, neue Ereignisse einfügen

import random
random.seed(startwert)                       # EINMAL, nicht in der Schleife
treffer = 0
for i in range(wiederholungen):
    if experiment():
        treffer = treffer + 1
print(treffer / wiederholungen)              # Schätzung der Wahrscheinlichkeit
```

- **Ereignisgesteuert** simulieren: Die Uhr springt von Ereignis zu Ereignis. Die Zahl der Schritte hängt von der **Zahl der Ereignisse** ab, nicht von der Länge der Zeit (ein Zähler zeigt es).
- In der **Ereignisliste** (`heapq`, Tupel `(zeit, art, nummer)`) sind die Ereignisse **immer der Zeit nach** geordnet, auch die erst später eingefügten. Bei **gleicher Zeit** entscheidet die Tupelreihenfolge
  (`art`, dann `nummer`). Lege sie bewusst fest und dokumentiere sie. Eine **negative Wartezeit** ist ein Hinweis auf Ereignisse in falscher Reihenfolge.
- **Kennzahlen** wie mittlere und größte Wartezeit vergleichen Varianten (A gegen B). Der Mittelwert kann besser werden, während das **Maximum** schlechter wird.
- Das **Monte-Carlo-Verfahren** schätzt eine Wahrscheinlichkeit durch viele Wiederholungen. Der **Startwert** (`random.seed`) wird **einmal** gesetzt und in der Dokumentation genannt, dann ist alles nachprüfbar.
- Die Schätzung **schwankt**: Der typische Fehler ist etwa `0,5 / Wurzel(N)`, hundertmal mehr Wiederholungen bringen zehnmal mehr Genauigkeit. Gib nicht mehr Stellen an, als sich begründen lassen.
- Wo es geht, **prüfe die Schätzung** mit der **genauen** Wahrscheinlichkeit durch **Zählen aller Fälle** (`2^(Spiele)`, nur für kleine Turniere).

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Fahrradwerkstatt](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/41_Fahrradwerkstatt/fahrradwerkstatt_loesung.ipynb) (41. BwInf): Arbeitszeit, zwei Verfahren, weitere Kennzahlen, Verfahren mit Höchstwartezeit
- [Tobis Turnier](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/39_TobisTurnier/tobisturnier_loesung.ipynb) (39.): Monte-Carlo für Liga und K.o., genaue Rechnung als Vergleich
- [Verzinkt](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/41_Verzinkt/verzinkt_loesung.ipynb) (41.): Kristallwachstum mit Ereignisliste, Pixelbilder und Zufall mit Startwert

Im nächsten Notebook (16) geht es um **Heuristiken**: Wenn man die beste Lösung nicht finden kann, verbessert man eine Lösung durch kleine Änderungen (lokale Suche) und startet mit Zufall immer wieder neu.